In [3]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.lines import Line2D

### process file

In [4]:
word_mapping = {
    'true': 1,
    'false': 0,
    'exec()': 0,
    'fork()': 1,
    'exit()': 2,
    'malloc()': 3,
    'free()': 4,
    'realloc()': 5,
    'ent:pthread_create()': 6,
    'ret:pthread_create()': 7,
    'ent:pthread_join()': 8,
    'ret:pthread_join()': 9,
    'ent:pthread_mutex_lock()': 10,
    'ret:pthread_mutex_lock()': 11,
    'ent:pthread_mutex_unlock()': 12,
    'ret:pthread_mutex_unlock()': 13
}

In [6]:
for k, v in word_mapping.items():
    print(f'{k}={v} ', end='')

true=1 false=0 exec()=0 fork()=1 exit()=2 malloc()=3 free()=4 realloc()=5 ent:pthread_create()=6 ret:pthread_create()=7 ent:pthread_join()=8 ret:pthread_join()=9 

In [5]:
def parse_key(key: str) -> int:
    if not key:
        return -1
    if key in word_mapping:
        return word_mapping[key]
    # bug: lifetime is a float, not handled correctly yet
    try:
        return int(key)
    except ValueError:
        return -1

def add_stats_line(arr: np.ndarray | None, line: list[str]) -> np.ndarray:
    if not line:
        return arr if arr is not None else np.empty((0, 0), dtype=int)

    vals = [parse_key(word) for word in line]
    row = np.array(vals, dtype=np.int64)[None, :] # trick to add a dimension to row

    if arr is None or arr.size == 0:
        #print('arr is None')
        return row
    if arr.dtype != np.int64:
        arr = arr.astype(np.int64, copy=False)
    if arr.ndim == 1: # if dimension of arr is 1
        #print('arr is dimension 1')
        arr = arr[None, :]
    if arr.shape[1] != row.shape[1]:
        raise ValueError(f'Column mismatch: arr has {arr.shape[1]} cols, row has {row.shape[1]} cols.')
    return np.vstack((arr,row))

In [ ]:
stats = None
with open("trace_results.csv", 'r') as trace_csv:
	trace_reader = csv.reader(trace_csv)
	next(trace_reader)

	for line in trace_reader:
		stats = add_stats_line(stats, line)

In [9]:
print(type(stats))
print(stats.shape)
print(stats.ndim)
print(stats[-5:])

<class 'numpy.ndarray'>
(130269, 8)
2
[[2570223624792         10655             1             4          9380
            152             4            -1]
 [2570223630824         10651             1             4          9340
            152            40            -1]
 [2570223636992         10644             1             4          9260
             72            80            -1]
 [2570223643189         10644             1             4          9252
             64             8            -1]
 [2570223927952         10644             1             2    2120208366
             -1            -1            -1]]


### ploting dynamic memory

In [10]:
# list format: timestamp, tid, isMain, operation, stat1, stat2, stat3, stat4
# operations: exec() marks the start of main thread, fork() marks start of child thread,
# 		malloc(), realloc(), free()
# ismain: 0 == False, 1 == True
# represent operations by int:
# 		true=1 false=0 exec()=0 fork()=1 exit()=2 malloc()=3 free()=4 realloc()=5 
# 		ent:pthread_create()=6 ret:pthread_create()=7 ent:pthread_join()=8 ret:pthread_join()=9 
# stats by operation:
# 		malloc(): stat1 = process heap size, stat2 = thread heap size, stat3 = size of malloc()
# 		realloc(): stat1 = process heap size, stat2 = thread heap size, stat3 = old size, stat4 = new size
# 		free(): stat1 = process heap size, stat2 = thread heap size, stat3 = size freed

In [ ]:
threads_mem = {}
program_mem = []
start_ms = stats[0][0] // 1000000
threads = set()
threads_pthread = defaultdict(lambda: ([], [])) #[pthread_create], [pthread_join]

for row in stats:
    tid = 0 if row[2] == 1 else row[1]
    # add threads
    if tid not in threads:
        threads.add(int(tid))
    if (row[3] == 0 or row[3] == 1) and tid not in threads_mem:
        threads_mem[tid] = ([0],[0]) #time_ns, memory_bytes
    if tid in threads_mem and row[3] in [3,4,5]: # traced calls
        timestamp = row[0] //1000000 - start_ms
        threads_mem[tid][0].append(timestamp)
        threads_mem[tid][1].append(row[5])
        program_mem.append((timestamp,row[4]))
    #trace pthread lib calls time used
    if row[3] == 7 or row[3] == 9:
        threads_pthread[tid][(row[3]-7)//2].append(row[5])

#dic.get(key, list())
print(f'All threads recorded are: {threads}.')
print(f'Relevant threads are: {[int(tid) for tid in threads_mem.keys()]}.')

In [26]:
lines = {}
plt.figure(figsize=(10,5))
plt.xlabel('time since program starts (ms)')
plt.ylabel('memory allocated (bytes)')
plt.title('dynamic memory by time')

# all threads dynamic memory plot
for tid, (time, mem) in threads_mem.items():
    smooth_window = 3
    # np.convolve with mode=valid only returns indices that can be convolved, so the first n-1 items in list is dropped
    # therefore add a total of n-1=9 paddings to mem to balance out the cut
    pad_left, pad_right = smooth_window // 2 - 1 if smooth_window % 2 == 0 else smooth_window // 2, smooth_window // 2
    mem_padded = np.pad(mem, (pad_left, pad_right), mode='edge')
    # each number is now the average of nth to n-9th number in mem
    mem_smooth = np.convolve(mem_padded, np.ones(smooth_window)/smooth_window, mode='valid')
    line, = plt.plot(time, mem_smooth, label=tid) # interesting unpack returned list of lines into a single line with ,
    lines[tid] = line
plt.legend([lines[0]], ["main"])
plt.savefig("plots/dynamic_memory.png")
plt.cla()

# selective threads dynamic memory plot
plt.plot(threads_mem[0][0], threads_mem[0][1])
child_line_count = 1
child_thread = list(threads_mem.keys())[child_line_count]
plt.plot(threads_mem[child_thread][0], threads_mem[child_thread][1])
plt.legend(["main", child_thread])
plt.savefig("plots/two_threads_memory.png")
plt.cla()

# process dynamic memory
plt.title('process heap memory by time')
time, mem = zip(*program_mem) #*iterable to unwrap
print(len(time), len(mem))
plt.plot(time, mem, label='process')
plt.savefig('plots/process_memory.png')
plt.clf()

plt.close()

All threads recorded are: {0, 10651, 10652, 10653, 10654, 10655}.
Relevant threads are: [0, 10651, 10652, 10653, 10654, 10655].
130257 130257


## no owner version

In [6]:
stats = None
with open("trace_no_owner_results.csv", 'r') as trace_csv:
	trace_reader = csv.reader(trace_csv)
	next(trace_reader)

	for line in trace_reader:
		stats = add_stats_line(stats, line)

In [7]:
print(type(stats))
print(stats.shape)
print(stats.ndim)
print(stats[-5:])

<class 'numpy.ndarray'>
(152039, 8)
2
[[1990331168024          6584             1            10            -1
             -1            -1            -1]
 [1990331179670          6584             1            -1         11646
             -1            -1            -1]
 [1990331186194          6584             1            12            -1
             -1            -1            -1]
 [1990331197319          6584             1            -1         11125
             -1            -1            -1]
 [1990331484601          6584             1             2    2505737526
             -1            -1            -1]]


In [8]:
threads_mem = {} # tid : ([time_ms], [memory_bytes])
program_mem = []
start_ms = stats[0][0] // 1000000
threads = set()

mem_used, with_realloc = 0, 0

for row in stats:
    tid = 0 if row[2] == 1 else row[1]
    # add threads
    if tid not in threads:
        threads.add(int(tid))
    if (row[3] == 0 or row[3] == 1) and tid not in threads_mem:
        threads_mem[tid] = ([],[]) #time_ms, memory_bytes
    if tid in threads_mem and row[3] in [3,4,5]: # malloc(), free(), realloc()
        if row[3] == 3:
            mem_used += row[6]
            with_realloc += row[6]
        if row[3] == 5:
            with_realloc += row[7] - row[6]
        timestamp = row[0] //1000000 - start_ms
        threads_mem[tid][0].append(timestamp)
        threads_mem[tid][1].append(row[5])
        program_mem.append((timestamp,row[4]))
print(f'{mem_used=}')
print(f'{with_realloc=}')
print(f'All threads recorded are: {threads}.')
print(f'Relevant threads are: {[int(tid) for tid in threads_mem.keys()]}.')

mem_used=np.int64(445187)
with_realloc=np.int64(445187)
All threads recorded are: {0, 6593, 6592, 6594, 6595, 6591}.
Relevant threads are: [0, 6591, 6592, 6593, 6594, 6595].


In [ ]:
lines = {}
plt.figure(figsize=(10,5))
plt.xlabel('time since program starts (ms)')
plt.ylabel('memory allocated (bytes)')
plt.title('dynamic memory by time')

# all threads dynamic memory plot
for tid, (time, mem) in threads_mem.items():
    smooth_window = 3
    # np.convolve with mode=valid only returns indices that can be convolved, so the first n-1 items in list is dropped
    # therefore add a total of n-1=9 paddings to mem to balance out the cut
    pad_left, pad_right = smooth_window // 2 - 1 if smooth_window % 2 == 0 else smooth_window // 2, smooth_window // 2
    mem_padded = np.pad(mem, (pad_left, pad_right), mode='edge')
    # each number is now the average of nth to n-9th number in mem
    mem_smooth = np.convolve(mem_padded, np.ones(smooth_window)/smooth_window, mode='valid')
    line, = plt.plot(time, mem_smooth, label=tid, alpha=0.8, lw=2) # interesting unpack returned list of lines into a single line with ,
    lines[tid] = line
plt.legend([lines[0]], ["main"])
#plt.yscale('symlog')
plt.savefig("plots/no_owner_dynamic_memory.png", dpi=300)
plt.cla()

# selective threads dynamic memory plot
plt.plot(threads_mem[0][0], threads_mem[0][1])
child_line_count = 1
child_thread = list(threads_mem.keys())[child_line_count]
plt.plot(threads_mem[child_thread][0], threads_mem[child_thread][1])
plt.legend(["main", child_thread])
plt.savefig("plots/no_owner_two_threads_memory.png", dpi=300)
plt.cla()

# process dynamic memory
plt.title('process heap memory by time')
plt.xlabel('time since program starts (ms)')
plt.ylabel('memory allocated (bytes)')
time, mem = zip(*program_mem) #*iterable to unwrap
print(len(time), len(mem))
plt.plot(time, mem, label='process')
plt.savefig('plots/no_owner_process_memory.png', dpi=300)
plt.clf()

plt.close()

#### different graph making techniques

In [10]:
lines = {}
plt.figure(figsize=(10,5))
plt.xlabel('time since program starts (ms)')
plt.ylabel('memory allocated (bytes)')
plt.title('dynamic memory by time')

# all threads dynamic memory plot
for tid, (time, mem) in threads_mem.items():
    smooth_window = 3
    # np.convolve with mode=valid only returns indices that can be convolved, so the first n-1 items in list is dropped
    # therefore add a total of n-1=9 paddings to mem to balance out the cut
    pad_left, pad_right = smooth_window // 2 - 1 if smooth_window % 2 == 0 else smooth_window // 2, smooth_window // 2
    mem_padded = np.pad(mem, (pad_left, pad_right), mode='edge')
    # each number is now the average of nth to n-9th number in mem
    mem_smooth = np.convolve(mem_padded, np.ones(smooth_window)/smooth_window, mode='valid')
    style = '-' if tid != 0 else '--'
    line, = plt.plot(time, mem_smooth, linestyle=style, label=tid, alpha=0.8, lw=2) # interesting unpack returned list of lines into a single line with ,
    lines[tid] = line
plt.legend([lines[0]], ["main"])
#plt.yscale('symlog')
plt.savefig("plots/thread_memories_dashed.png", dpi=300)
plt.cla()

for tid, (time, mem) in threads_mem.items():
    smooth_window = 3
    # np.convolve with mode=valid only returns indices that can be convolved, so the first n-1 items in list is dropped
    # therefore add a total of n-1=9 paddings to mem to balance out the cut
    pad_left, pad_right = smooth_window // 2 - 1 if smooth_window % 2 == 0 else smooth_window // 2, smooth_window // 2
    mem_padded = np.pad(mem, (pad_left, pad_right), mode='edge')
    # each number is now the average of nth to n-9th number in mem
    mem_smooth = np.convolve(mem_padded, np.ones(smooth_window)/smooth_window, mode='valid')
    marker = '' if tid != 0 else 'o'
    line, = plt.plot(time, mem_smooth, marker=marker, label=tid, alpha=0.8, lw=2) # interesting unpack returned list of lines into a single line with ,
    lines[tid] = line
plt.legend([lines[0]], ["main"])
#plt.yscale('symlog')
plt.savefig("plots/thread_memories_markers.png", dpi=300)
plt.cla()

# process dynamic memory
plt.title('process heap memory by time')
plt.xlabel('time since program starts (ms)')
plt.ylabel('memory allocated (bytes)')
time, mem = zip(*program_mem) #*iterable to unwrap
print(len(time), len(mem))
plt.plot(time, mem, label='process')
# draw vertical lines whenever a thread ends
for tid in threads_mem.keys():
    if tid == 0:
        continue
    plt.axvline(x=threads_mem[tid][0][-1], linestyle="--")
# create proxy lines for custom legend
legend_elements = [
    Line2D([0], [0], linestyle='--', label='thread ends'),
    Line2D([0], [0], linestyle='-', label='process heap')
]
plt.legend(handles=legend_elements, loc='upper left')
plt.savefig('plots/process_memory_threads_end.png', dpi=300)
plt.clf()

plt.close()

130259 130259


### pthread lib

In [39]:
# [uprobe create time, uretprobe create time, create time used, uprobe join time, uretprobe join time, join time used]
# one for each thread
create_enter = []
create_end = []
create_durations = []
join_enter = []
join_end = []
join_durations = []
start_ns = stats[0][0]

for row in stats:
    tid = row[1]
    if row[3] == 6:
        create_enter.append(row[0] - start_ns)
    if row[3] == 7:
        create_end.append(row[0] - start_ns)
        create_durations.append(row[5])
    if row[3] == 8:
        join_enter.append(row[0] - start_ns)
    if row[3] == 9:
        join_end.append(row[0] - start_ns)
        print(f'join: {row[5]=}, {row[0]-start_ns=}')
        join_durations.append(row[5])

join: row[5]=np.int64(571116229), row[0]-start_ns=np.int64(572827615)
join: row[5]=np.int64(1047123262), row[0]-start_ns=np.int64(1619959554)
join: row[5]=np.int64(269585538), row[0]-start_ns=np.int64(1889553240)
join: row[5]=np.int64(14316), row[0]-start_ns=np.int64(1889574367)
join: row[5]=np.int64(184673875), row[0]-start_ns=np.int64(2074251745)


In [40]:
# box plot for time used in each function

fig, axes = plt.subplots(2, 1, sharex=False)

axes[0].boxplot(create_durations, vert=False)
axes[0].set_title('pthread_create()')

axes[1].boxplot(join_durations, vert=False)
axes[1].set_title('pthread_join()')

plt.xlabel("time (ns)")
plt.tight_layout()
plt.savefig('plots/pthread_functions_time.png', dpi=300)
plt.close()

In [ ]:
tid_lock_map = defaultdict(lambda : ([], [], [], [])) # [uprobe:lock], [uretprobe:lock], [uprobe:unlock], [uretprobe:unlock]
start_ns = starts[0][0]
for row in stats:
    tid = row[1]
    if row[3] == 10: # uretprobe:libc:pthread_mutex_lock()
        tid_lock_map[tid][0].append(row[0] - start_ns)
    elif row[3] == 11:
        tid_lock_map[tid][1].append(row[0] - start_ns)
    elif row[3] == 12:
        tid_lock_map[tid][2].append(row[0] - start_ns)
    elif row[3] == 13:
        tid_lock_map[tid][3].append(row[0] - start_ns)

tid_lock_hold = 